# Backbone probe: `eva02l`

**EVA-02 ViT-L/14 @ 448 (timm)**

A third pretraining PARADIGM rather than a fourth ViT: masked image modelling distilled from a CLIP teacher, then supervised fine-tuning on IN-22k/IN-1k. Neither DINOv2's self-distillation nor SigLIP's contrastive image-text objective.

One model per notebook, on purpose. There is no BACKBONE knob here: an arm is
only comparable with the others if the manifest, the standardisation policy and
the seed are identical across all four, and the cheapest way to guarantee that
is to leave nothing switchable. Run the other three from their own notebooks,
in parallel sessions.

| | notebook | tower | projected |
|---|---|---|---|
|  | `dinov2regl` | DINOv2 with registers, ViT-L/14 @ 518 | ~120 min |
| **this one** | `eva02l` | EVA-02 ViT-L/14 @ 448 (timm) | ~90 min |
|  | `convnextv2h` | ConvNeXt V2 Huge @ 384 | ~135 min |
|  | `siglipso400m` | SigLIP SO400M/14 @ 384 | ~120 min |

## What this arm has to watch

This is the only arm whose input is DOWNSAMPLED. EVA-02's pretrained_cfg sets fixed_input_size, so the tower admits exactly 448 while canonicalise emits a 512 nominal side. It is also the cheapest arm, at 1024 tokens against dinov2regl's 1369 -- the handicap and the discount are one fact. Quote both together.

It is also the only arm loaded through timm. Kaggle's image ships timm, so the install cell has nothing to do; if it is ever missing, load_backbone fails with the install line rather than a bare ImportError.

## Held fixed across all four arms

- **The same 20,000-row probe manifest**, fingerprint `fdb2c38eab6b8664...`,
  asserted in the verify cell. A different digest is a different 20,000 rows
  and the arm is not comparable with anyone else's.
- **`CANON_MODE = "band"`**, matching the `dinov3l` and `dinov2l` bars. This is
  a BACKBONE comparison; standardisation is a separate question with its own
  probe.
- **`SEED = 20260827`**, which is what makes the augmented views bit-identical
  across arms.

## Before you start

- Settings -> Accelerator -> **GPU T4 x2** (or P100).
- Settings -> Internet -> **On**.
- Add data -> **`techjam-aigc-probe-union`**. One Dataset, and only one: it carries the training
  images, the eval subsample, `demo/` and both manifests.
- No HuggingFace token. All four candidates are ungated (`docs/model_licences.md`).

**Do not publish the bank this produces as a Dataset.** A probe manifest
fingerprints differently from the real one, so a probe bank cannot verify,
merge, resume or fuse against a real bank -- every one of those refusals is
correct. Download `selection_probe_band_eva02l.json` instead; it is kilobytes.

## 0. Parameters

In [ ]:
# ============ ONE MODEL, NO BACKBONE KNOB ============
BACKBONE = "eva02l"          # this notebook is eva02l and nothing else
SMOKE    = True           # True first: proves the chain in minutes
PHASES   = "auto"         # "auto", or e.g. "stage_a" / "eval_bank,ladder"
# =====================================================

# The probe corpus and its standardisation policy travel together, and neither
# is a free knob: band mode is what the dinov3l bars were measured under.
CANON_MODE = "band"
SEED       = 20260827
SPLITS     = "train,val_internal"
RUNGS      = ['a0', 'a1', 'a2', 'a3', 'a7_norecon']
TIER       = "ablation"

# Fingerprints of the two probe manifests, pinned. Asserted in the verify cell
# below -- see the header for why an unpinned probe silently ruins the arm.
EXPECT_TRAIN_SHA = "fdb2c38eab6b8664fb5043d47df0c8d65f383a6123e4d9c5e5ebcb15c60570e3"
EXPECT_EVAL_SHA  = "7f863cfcca12f4e76ea8cc3e64ea9d11ccac1246e5c9d537eef04e135d98c9b0"

DATASET_SLUG = "techjam-aigc-probe-union"
# Recursive, because Kaggle mounts at either /kaggle/input/<slug> or
# /kaggle/input/datasets/<owner>/<slug>.
MANIFEST_GLOB      = f"/kaggle/input/**/{DATASET_SLUG}*/manifest_union_probe.parquet"
EVAL_MANIFEST_GLOB = f"/kaggle/input/**/{DATASET_SLUG}*/eval_manifest_union_probe.parquet"
# No separate DATA_GLOB. The images live in the SAME Dataset as the manifests,
# so the mount is the manifest's own directory -- derived rather than globbed
# for a second time, which is one fewer pattern that can match a different
# number of things than the first one did.

WORKERS          = 4
BATCH_SIZE       = 16
CHECKPOINT_EVERY = 200            # images between flushes = work at risk

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

WORK      = "/kaggle/working"
BANK_DIR  = f"{WORK}/banks/probe_band_{BACKBONE}"
EVAL_DIR  = f"{WORK}/banks/eval_probe_band_{BACKBONE}"
RUNS_DIR  = f"{WORK}/outputs/rungs_probe_band_{BACKBONE}"
DOCS_DIR  = f"{WORK}/outputs/docs"
TABLE     = f"{DOCS_DIR}/robustness_table_probe_band_{BACKBONE}.md"
SELECTION = f"{DOCS_DIR}/selection_probe_band_{BACKBONE}.json"

print(f"arm: {BACKBONE}  canon={CANON_MODE}  seed={SEED}  smoke={SMOKE}")

## 1. Get the code

Public repo, shallow clone, and a `reset --hard` on re-run so a resumed session
picks up any fix without a stale working tree. Nothing here is authenticated.

In [ ]:
import glob, importlib, os, subprocess, sys, time

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

# Two directories, for two different importers. `notebooks/` is where
# `kaggle_bootstrap` lives. `src/` is where the `aigcdet` package lives, and it
# is put on the path HERE rather than left to the `pip install -e` in the next
# section, because an editable install registers itself through a .pth file
# that site.py reads at INTERPRETER START. A kernel that was already running
# when pip finished never sees it. Every extraction is a subprocess with a
# fresh interpreter, so those work either way -- but an in-kernel
# `from aigcdet...` raises ModuleNotFoundError, several cells later, with the
# install cell reporting success.
for _sub in ("notebooks", "src"):
    _p = os.path.join(REPO_DIR, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
importlib.invalidate_caches()
import kaggle_bootstrap as kb
importlib.reload(kb)

sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
print("helper loaded from", kb.__file__)

## 2. Install — without losing Kaggle's torch

This is the step that ends sessions. `pip install -e .` hands pip the
`torch>=2.0` line from `pyproject.toml` and invites it to resolve a torch built
for a different CUDA than this machine's drivers; you get a `torch` that cannot
see the GPU and no way back except a factory reset.

So: the project goes in with `--no-deps` (a pure path registration, which is all
it is needed for), everything else is installed **only if genuinely missing**,
and nothing CUDA-matched is touched at all. `transformers` is the one exception
— Kaggle images move and the project needs ≥4.53 for DINOv3 — and it is
upgraded with `--no-deps` too.

**Print the plan before running it.** If you ever see `torch` in that list,
stop.

In [ ]:
def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))

print("pip plan:")
for cmd in plan:
    print("   ", " ".join(cmd))

assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for cmd in plan for w in cmd), "STOP: the plan would touch torch"

for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("\ninstall done")

# Prove the project is importable IN THIS KERNEL, here, where the remedy is
# still "re-run the two cells above". Without this the first in-kernel
# `from aigcdet...` is in the auth section, and a ModuleNotFoundError there
# reads as an auth problem rather than a path one.
importlib.invalidate_caches()
from aigcdet.features.backbones import BACKBONES as _B
print("aigcdet importable:", len(_B), "backbones registered")

### 2b. Check the environment before paying for anything

Every problem this cell can report makes the 1.2 GB model download pointless,
so it runs before the download rather than after it.

If it tells you `transformers` was upgraded: **restart the kernel**
(Run → Restart session) and re-run from cell 0. A module already imported at
the old version stays imported.

In [ ]:
import platform

torch_v = installed_version("torch")
tf_v    = installed_version("transformers")
problems = kb.environment_problems(platform.python_version(), torch_v, tf_v)

print(f"python {platform.python_version()}  torch {torch_v}  transformers {tf_v}")
if problems:
    for p in problems:
        print("\nPROBLEM:", p)
    raise SystemExit("fix the above before continuing")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU, then restart the session. "
    "Extraction on CPU will not finish inside a session.")

## 3. HuggingFace auth

**Only if your `BACKBONE` is gated.** SigLIP2 (Apache-2.0) and CLIP (MIT) are
public: the cell below will say so and move on, and you need no token at all.
DINOv3 is gated behind Meta's licence, and then two separate things must be
true — from this notebook they fail identically, as a 401/403 on
`from_pretrained`:

1. **Your own** HuggingFace account has accepted the licence at the model page.
   Acceptance is per account — the project owner's acceptance does nothing for
   yours.
2. A read token from that same account is attached to this notebook as a Kaggle
   Secret named `HF_TOKEN`.

**Never paste a token into a cell.** This repo is public and a notebook is
committed with its cell source. Add-ons → Secrets is the whole reason that
mechanism exists.

In [ ]:
# Both read off the registry, so they follow BACKBONE rather than being
# typed again here. DINOv3 is gated behind Meta's licence; SigLIP2
# (Apache-2.0) and CLIP (MIT) are not, and the fleet should not be
# stopped for a token its run never uses.
from aigcdet.features.backbones import BACKBONES
MODEL_ID = BACKBONES[BACKBONE].hf_id
GATED    = kb.requires_hf_token(BACKBONE)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

token = kb.hf_token(secrets)
for line in kb.hf_auth_advice(token, MODEL_ID, gated=GATED):
    print(line)

if token:
    # Exported for `transformers` to pick up. Never printed, never written to a
    # file that leaves this session.
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
elif GATED:
    raise SystemExit("no HuggingFace token -- see the instructions above")

## 4. Attach the data, and prove it is intact -- do not skip this

One mount. `unify_mounts` locates the corpus root by the top-level names the
manifest itself reports, so Kaggle's wrapper directories do not have to be
guessed at.

In [ ]:
import pandas as pd

MANIFEST = sorted(glob.glob(MANIFEST_GLOB, recursive=True))
EVAL_MANIFEST_PATHS = sorted(glob.glob(EVAL_MANIFEST_GLOB, recursive=True))
assert MANIFEST, f"no probe Dataset attached (looked for {MANIFEST_GLOB})"
assert len(MANIFEST) == 1, f"{len(MANIFEST)} manifests match: {MANIFEST}"
assert len(EVAL_MANIFEST_PATHS) == 1, f"eval manifests: {EVAL_MANIFEST_PATHS}"
MANIFEST, EVAL_MANIFEST = MANIFEST[0], EVAL_MANIFEST_PATHS[0]
# Kaggle mounts at either /kaggle/input/<slug> or
# /kaggle/input/datasets/<owner>/<slug>, and the recursive glob above already
# found whichever it is. Taking the dirname cannot disagree with it.
DATA_MOUNTS = [os.path.dirname(MANIFEST)]
print("train manifest:", MANIFEST)
print("eval manifest: ", EVAL_MANIFEST)

# What the manifest says its root contains -- this locates the root inside the
# mount instead of guessing at Kaggle's wrapper directories.
EXPECTED = kb.top_level_names(pd.read_parquet(MANIFEST, columns=["rel_path"]))
print("dataset root should contain:", sorted(EXPECTED))

UNIFIED = kb.unify_mounts(DATA_MOUNTS, "/kaggle/temp/aigcdet_probe_root", EXPECTED)
DATA_ROOT = UNIFIED.root
print("unified root:", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

# The EVAL manifest is rooted one level up -- its rel_paths start with `demo/`
# or `normalized_union/`. Both are top-level entries of this same Dataset, so
# unlike the full streams there is no second mount and no two-link farm: the
# unified root already resolves both. Proven on real rows below rather than
# assumed, because a farm can list correctly and resolve to nothing.
EVAL_ROOT = DATA_ROOT
_rel = pd.read_parquet(EVAL_MANIFEST, columns=["rel_path"])["rel_path"]
_missing = [x for x in _rel.sample(200, random_state=SEED)
            if not os.path.exists(os.path.join(EVAL_ROOT, x))]
assert not _missing, (
    f"{len(_missing)} of 200 sampled eval rows do not resolve under "
    f"{EVAL_ROOT}, e.g. {_missing[:3]}")
print("200 sampled eval rows all resolve")

In [ ]:
t0 = time.time()
manifest, GATE = kb.open_verified_manifest(
    MANIFEST, DATA_ROOT,
    sample=2000 if SMOKE else None,
    # os.walk does not follow the farm's symlinked directories, so an
    # "extra files: 0" from it would be unearned. Skipped and said so.
    check_extra=not UNIFIED.linked,
)
print(kb.describe_gate(GATE))

# THE PIN. Every arm of this probe must be scored on the same 20,000 rows, and
# a re-cut probe manifest is the failure that produces a number which looks
# fine and is not comparable with anyone else's. Two minutes here against a
# two-hour arm that has to be thrown away.
assert GATE.manifest_sha256 == EXPECT_TRAIN_SHA, (
    f"probe manifest fingerprint is {GATE.manifest_sha256}, expected "
    f"{EXPECT_TRAIN_SHA}. This is a DIFFERENT 20,000 rows, so the arm would "
    f"not be comparable with the other three or with the dinov3l bars. "
    f"Re-cut it with scripts/cut_probe_manifest.py, or attach the published "
    f"techjam-aigc-probe-union Dataset.")

from aigcdet.features.bank import manifest_fingerprint
_eval_sha = manifest_fingerprint(pd.read_parquet(EVAL_MANIFEST))
assert _eval_sha == EXPECT_EVAL_SHA, (
    f"eval manifest fingerprint is {_eval_sha}, expected {EXPECT_EVAL_SHA}")

print(f"\nboth manifests match the pinned fingerprints")
print(f"verified in {time.time() - t0:.0f}s")
print(manifest["split"].value_counts().to_string())

## 5. What this arm actually loads

Printed from the registry rather than typed here, and the **effective** dtype
rather than the declared one -- a `bfloat16` spec on a T4 or P100 falls back to
`float32` and runs about 3x slower, which is worth seeing now and not at hour
two.

In [ ]:
from aigcdet.features.backbones import BACKBONES, run_dtype

spec = BACKBONES[BACKBONE]
DIM = spec.dim
print(f"{spec.name}: {spec.hf_id}")
print(f"  image_size {spec.image_size}   dim {DIM}   "
      f"prefix_tokens {spec.num_prefix_tokens}   pool {spec.pool}")
print(f"  params {spec.params:,}   gated {spec.gated}   loader {spec.loader}")
print(f"  normalisation mean={spec.mean} std={spec.std}")
print(f"  dtype declared {spec.dtype}, EFFECTIVE on this GPU "
      f"{run_dtype(spec, 'cuda')}")
if run_dtype(spec, "cuda") != spec.dtype:
    print("  ! this GPU has no hardware for the declared dtype; expect ~3x "
          "the projected time")

# 20,000 rows is one shard on any of these. Asserted rather than assumed: a
# bank that overflows /kaggle/working dies at the flush, hours in.
n_rows = int(len(kb.select_splits(manifest, SPLITS)))
ok, why = kb.fits_in_working(n_rows, DIM, n_views=11)
print(f"\n{n_rows} rows x 11 views x dim {DIM}: {why}")
assert ok, why

## 6. Which phases this session runs

Resumable, because a two-hour arm that dies at 100 minutes should not restart
from zero. Re-running every cell from the top continues where it stopped.

In [ ]:
import json

def _bank_done(d):
    """A bank is complete when its metadata says every row was written."""
    st = kb.read_resume_state(d)
    return st.exists and st.n_images > 0 and st.n_done >= st.n_images

STATUS = {
    "stage_a":   _bank_done(BANK_DIR),
    "eval_bank": _bank_done(EVAL_DIR),
    "ladder":    os.path.exists(SELECTION),
}
ORDER = ["stage_a", "eval_bank", "ladder"]
TODO = ([p for p in ORDER if not STATUS[p]] if PHASES == "auto"
        else [p.strip() for p in PHASES.split(",") if p.strip()])

for p in ORDER:
    mark = "done" if STATUS[p] else ("TODO" if p in TODO else "skip")
    print(f"  {p:10s} {mark}")
print(f"\nthis session will run: {TODO or '(nothing)'}")

## 7. Smoke run

Two timed runs, then the marginal rate between them -- the first includes the
model download and would flatter nothing if used alone. The projection for this
arm is **~90 minutes**; read the measured number below rather
than trusting it.

In [ ]:
def stage_a_argv(limit=None):
    return kb.run_shard_argv(
        GATE, manifest_path=MANIFEST, root=DATA_ROOT, backbone=BACKBONE,
        out_dir=BANK_DIR, splits=SPLITS, shard=0, n_shards=1,
        resume=True, workers=WORKERS, batch_size=BATCH_SIZE,
        checkpoint_every=CHECKPOINT_EVERY, limit=limit,
        canon_mode=CANON_MODE)

if SMOKE and "stage_a" in TODO:
    import shutil
    timings = {}

    def timed_smoke(n):
        # A throwaway directory: a smoke bank of 8 rows would otherwise be
        # resumed from as if it were the real one.
        out = f"/kaggle/temp/smoke_{BACKBONE}_{n}"
        shutil.rmtree(out, ignore_errors=True)
        argv = [a if a != BANK_DIR else out for a in stage_a_argv(limit=n)]
        t = time.time()
        rc = kb.run_streaming(argv)
        assert rc == 0, f"smoke run of {n} exited {rc}"
        timings[n] = time.time() - t
        print(f"  {n} images in {timings[n]:.0f}s")
        return timings[n]

    small, large = 8, 40
    timed_smoke(small)
    timed_smoke(large)
    # Run 2 reads the model from cache, so a cheap tower can measure a
    # NEGATIVE marginal rate and marginal_rate refuses it -- correctly.
    # Escalate the large sample until the two timings separate.
    ceiling = 8 * large
    while True:
        try:
            RATE = kb.marginal_rate(small, timings[small], large, timings[large])
            break
        except ValueError as e:
            if large >= ceiling:
                raise SystemExit(
                    f"could not measure a marginal rate up to {large} images: "
                    f"{e}") from None
            print(f"\n  {e}\n  -> doubling the large sample and re-measuring")
            large *= 2
            timed_smoke(large)

    n_rows = int(len(kb.select_splits(manifest, SPLITS)))
    plan = kb.session_plan(n_rows, RATE, checkpoint_every=CHECKPOINT_EVERY)
    print(f"\n{RATE:.3f} s/image marginal (model download excluded)")
    print(f"stage A over {plan.n_images} images -> {plan.hours:.1f} h, "
          f"{plan.sessions_needed} session(s), "
          f"{plan.minutes_at_risk:.0f} min at risk per kill")
    for note in plan.notes:
        print("  !", note)
    print("\nSet SMOKE = False and re-run from the top to start the real arm.")
else:
    print("smoke: skipped")

## 8. Stage A

In [ ]:
if "stage_a" in TODO and not SMOKE:
    argv = stage_a_argv()
    print(" ".join(argv[1:]), "\n")
    t0 = time.time()
    rc = kb.run_streaming(argv)
    assert rc == 0, f"stage A exited {rc}"
    print(f"\nstage A finished in {(time.time() - t0)/60:.1f} min")
    st = kb.read_resume_state(BANK_DIR)
    print(f"{st.n_done}/{st.n_images} images ({st.fraction_done:.1%})")
elif SMOKE:
    print("stage_a: SMOKE is still True")
else:
    print("stage_a: skipped")

## 9. Is the bank finite?

**Do not skip this cell.** On 2026-08-29 a five-hour DINOv3 bank came back
131,116 rows of NaN -- produced at full speed, with nothing raising, and the
only post-condition checked was the row count. `float16` overflow is silent.

For `eva02l` specifically, see "What this arm has to watch" at the top.

In [ ]:
if not SMOKE and _bank_done(BANK_DIR):
    import numpy as np

    from aigcdet.features.bank import FeatureBank

    bank = FeatureBank(BANK_DIR)
    bank.check_invariants()

    # Sampled evenly rather than from the head: an overflow that begins
    # part-way through a corpus (a brighter source, a larger image) leaves the
    # first rows finite.
    feats = np.load(os.path.join(BANK_DIR, "feats.npy"), mmap_mode="r")
    idx = np.linspace(0, feats.shape[0] - 1, min(4096, feats.shape[0])).astype(int)
    sample = np.asarray(feats[idx], dtype=np.float32)
    n_bad = int((~np.isfinite(sample)).sum())
    print(f"sampled {len(idx)} of {feats.shape[0]} rows: "
          f"{n_bad} non-finite values, max|x| {np.abs(sample).max():.2f}")
    assert n_bad == 0, (
        f"{n_bad} non-finite values in the bank. This is a DTYPE problem, not "
        f"a bad image. Fix BackboneSpec.dtype for {BACKBONE} (bfloat16, or "
        f"float32 on a GPU without it) and extract to a NEW directory -- do "
        f"not delete this one from under a resume.")
    print("bank is finite")
else:
    print("no completed bank to check yet")

## 10. The evaluation bank

4,000 rows x 20 conditions. `--no-subsample`: the probe eval manifest is
already the cut, and subsampling it again would score a different set of rows
in each arm.

In [ ]:
if "eval_bank" in TODO and not SMOKE:
    argv = [sys.executable, f"{REPO_DIR}/scripts/extract_eval_bank.py",
            "--manifest", EVAL_MANIFEST, "--backbone", BACKBONE,
            "--out", EVAL_DIR, "--tier", TIER, "--root", EVAL_ROOT,
            "--device", "cuda", "--batch-size", str(BATCH_SIZE),
            "--seed", str(SEED),
            "--checkpoint-every", str(CHECKPOINT_EVERY), "--resume",
            "--no-subsample", "--canon-mode", CANON_MODE]
    print(" ".join(argv[1:]), "\n")
    rc = kb.run_streaming(argv)
    assert rc == 0, f"eval bank exited {rc}"
else:
    print("eval_bank: skipped")

## 11. The ladder

In [ ]:
if "ladder" in TODO and not SMOKE:
    os.makedirs(DOCS_DIR, exist_ok=True)
    rung_cfgs = [f"{REPO_DIR}/configs/rungs/{r}.yaml" for r in RUNGS]
    for c in rung_cfgs:
        assert os.path.exists(c), f"missing rung config {c}"

    argv = [sys.executable, f"{REPO_DIR}/scripts/run_ablation.py",
            "--bank", BANK_DIR, "--eval-bank", EVAL_DIR,
            "--rungs", *rung_cfgs,
            "--tier", TIER, "--device", "cuda",
            "--out", TABLE, "--selection", SELECTION,
            "--heatmap", f"{DOCS_DIR}/robustness_heatmap_probe_band_{BACKBONE}.png",
            "--out-dir", RUNS_DIR]
    print(" ".join(argv[1:]), "\n")
    rc = kb.run_streaming(argv)
    assert rc == 0, f"ladder exited {rc}"
else:
    print("ladder: skipped")

## 12. Read the result

`heldout_robust_tpr_at_1pct`, at the same rung, against the frozen `dinov3l`
ladder. **Not clean AUC** -- degraded generalisation is what this project
selects on.

Two caveats travel with every number below, and neither is optional:

- **SCALE.** 20,000 rows against the corpus's 375,358. This ranks candidates;
  it does not settle the shipped choice.
- **POPULATION.** The dinov3l bars were measured on the frozen 138,116-row
  corpus, which is a different population from the union probe. A gap of a few
  points against them is not, on its own, evidence about the tower.

In [ ]:
bars = {'a0': 0.8611, 'a1': 0.9037, 'a2': 0.9037, 'a3': 0.9012, 'a7_norecon': 0.0296}

if os.path.exists(SELECTION):
    sel = json.load(open(SELECTION))
    print(f"arm: {BACKBONE}  (dim {DIM}, {spec.params:,} params, "
          f"{run_dtype(spec, 'cuda')})")
    print(f"headline: {sel.get('headline')}")
    print(f"metric:   {sel.get('metric')}\n")
    print(f"{'rung':14s} {'this arm':>10s} {'dinov3l':>10s} {'delta':>10s}")
    for rung, v in sorted(sel.get("summary", {}).items()):
        got = v.get("heldout_robust_tpr_at_1pct")
        bar = bars.get(rung)
        if got is None:
            continue
        delta = f"{got - bar:+.4f}" if bar is not None else "--"
        print(f"{rung:14s} {got:10.4f} "
              f"{(f'{bar:.4f}' if bar is not None else '--'):>10s} {delta:>10s}")
    print(f"\ntable: {TABLE}")
    print("\nSCALE:      20,000 rows vs the corpus's 375,358.")
    print("POPULATION: the dinov3l bars are from the frozen 138,116-row corpus.")
    print("\nDownload the selection JSON; do not publish the bank as a Dataset.")
else:
    print("no ladder output yet -- run the remaining phases in a new session")

for d in (BANK_DIR, EVAL_DIR):
    st = kb.read_resume_state(d)
    if st.exists:
        print(f"  {d}: {st.n_done}/{st.n_images} ({st.fraction_done:.0%})")

## The 2am playbook

| What you see | What to do |
|---|---|
| `CUDA out of memory` | Lower `BATCH_SIZE` to 8, then 4. **Retryable** -- nothing repeats. |
| `MemoryError`, kernel dies | Lower `WORKERS`. Retryable. |
| `ReadTimeout`, `ConnectionError` | The clone or the model download. Just re-run. |
| non-finite values in the bank | **Fatal for this dtype.** Extract to a NEW directory after fixing `BackboneSpec.dtype`; do not delete the old one from under a resume. |
| `probe manifest fingerprint is ...` | **Fatal.** You attached a different probe cut. This arm would not be comparable. |
| `cannot resume the bank at ...` | **Fatal.** A parameter moved between sessions. Restore it, or extract to a new directory. |
| `no probe Dataset attached` | You attached the wrong Dataset, or the share has not reached you. |
| `gated repo` / `401` / `403` | Should be impossible here -- all four candidates are ungated. It means the mirror, not auth. |

Do **not**: change `BACKBONE`, `SEED`, `CANON_MODE` or `SPLITS` between
sessions of the same arm; delete a bank that refuses to resume; or skip the
finite-check cell. A `pip install torch` would replace Kaggle's driver-matched
build and cost the session.

Paste an error below and it will say whether re-running can possibly help.

In [ ]:
ERROR = """paste the error here"""
print(kb.explain(ERROR))